Extract Data from the Full Catchment Attributes Dataset

In [ ]:
import duckdb

# con = duckdb.connect("../../data/database/database.duckdb", read_only=True)

# df = con.execute("SELECT * FROM main_marts.gauge_streamflow_availability").df()
# con.close()

# Remove stations with zero data availability
# df = df[df["data_availability [hrs]"] > 0]


In [ ]:
con = duckdb.connect("../../data/database/database.duckdb", read_only=True)

ca_df = con.execute("SELECT * FROM main_marts.dim_catchment_attributes_full").df()
con.close()

# Limit the sites based on the Missouri River Basin HUC 10 units (10U and 10L)
ca_df = ca_df[ca_df["HUC02"].isin(["10U", "10L"])]

# Limit the sites based on the state of Missouri
limit_missouri = False

if limit_missouri == True:
    ca_df = ca_df[ca_df["state_gage"] == "MO"]

print(ca_df.shape)

(923, 559)


In [3]:
# Define Flooding
ca_df["discharge_range"] = ca_df["discharge_max"] - ca_df["discharge_min"]

In [6]:
# Merge the two dataframes to get the final set of stations to used that have data.
# ca_df = ca_df[ca_df["site_id"].isin(df["STAID"])]



print(ca_df.shape)

# Determine how many stations are in each HUC unit.
print(ca_df["state_gage"].value_counts().sort_index())
print(ca_df.head())


(923, 560)
state_gage
CO     83
IA     36
KS     97
MO    100
MT    139
ND     54
NE    141
SD    122
WY    151
Name: count, dtype: int64
      site_id                          STANAME HUC02   LAT_GAGE   LNG_GAGE  \
773  06466400    Bazile Creek at Center, Nebr.   10L  42.616389 -97.878056   
774  06470000     JAMES RIVER AT JAMESTOWN, ND   10U  46.889722 -98.681667   
775  06470500       JAMES RIVER AT LAMOURE, ND   10U  46.355525 -98.304543   
776  06470878  JAMES RIVER AT ND-SD STATE LINE   10U  45.936358 -98.174269   
777  06471000           JAMES R AT COLUMBIA SD   10U  45.603580 -98.310376   

    state_gage  FIPS_SITE COUNTYNAME_SITE  DRAIN_SQKM agg_ecoregion  ...  \
773         NE      31107            Knox     816.462    CntlPlains  ...   
774         ND      38093        Stutsman    7459.441    CntlPlains  ...   
775         ND      38045        La Moure   11189.830    CntlPlains  ...   
776         SD      38021          Dickey   13773.150    CntlPlains  ...   
777         S

In [ ]:
import plotly.express as px

fig = px.scatter_geo(
    ca_df,
    lat="LAT_GAGE",
    lon="LNG_GAGE",
    color="HUC02",
    scope="usa",
    hover_name="site_id",
    hover_data=["STANAME"],
    title="Streamflow Gauge Locations by HUC02 Region",
    labels={"HUC02": "HUC02 Region"},
)

fig.update_layout(geo=dict(
    showland=True,
    showlakes=True,
    showsubunits=True,
    subunitcolor="gray",
))

fig.update_layout(
    title_x=0.5,
    margin={"r": 0, "t": 30, "l": 20, "b": 0}
)

fig.show()


In [ ]:
import plotly.express as px

fig = px.scatter_geo(
    ca_df,
    lat="LAT_GAGE",
    lon="LNG_GAGE",
    color="p_mean",
    color_continuous_scale="Viridis",
    scope="usa",
    hover_name="site_id",
    hover_data=["STANAME"],
    title="Average Precipitation at Streamflow Gauge Locations",
    labels={"p_mean": "Mean Precipitation (m)"},
)

fig.update_layout(geo=dict(
    showland=True,
    showlakes=True,
    showsubunits=True,
    subunitcolor="gray",
))

# For the State of Missouri
if limit_missouri == True:
    fig.update_geos(
        scope="usa",
        center={"lat": 38.5, "lon": -92.5},
        projection_scale=5.8  # increase to zoom in more
    )

fig.update_layout(
    title_x=0.5,
    margin={"r": 0, "t": 30, "l": 20, "b": 0}
)


fig.show()


In [ ]:
def plot_gauge_locations(df, title_text, color_variable, colorscale_text, zmin, zmax):
    fig = px.scatter_geo(
        df,
        lat="LAT_GAGE",
        lon="LNG_GAGE",
        color=color_variable,
        range_color=(zmin, zmax),
        color_continuous_scale="Viridis",
        scope="usa",
        hover_name="site_id",
        hover_data=["STANAME", "COUNTYNAME_SITE"],
        title=title_text,
        labels={color_variable: colorscale_text},
    )

    fig.update_layout(geo=dict(
        showland=True,
        showlakes=True,
        showsubunits=True,
        subunitcolor="gray"
    ))

    fig.update_layout(
        title_x=0.5,
        margin={"r": 0, "t": 30, "l": 20, "b": 0}
)

  # For the State of Missouri
    if limit_missouri == True:
        fig.update_geos(
            scope="usa",
            center={"lat": 38.5, "lon": -92.5},
            projection_scale=5.8  # increase to zoom in more
        )

    fig.show()

In [ ]:
# Plot the gauge locations colored by the discharge range (flooding severity)
plot_gauge_locations(ca_df, "Flooding Severity", "discharge_range", "Discharge Range (m³/s)", 0, 85)

In [ ]:
# Plot the gauge locations colored by the maximum discharge (flooding severity)
plot_gauge_locations(ca_df, "Flooding Severity", "discharge_max", "Discharge Max (m³/s)", 0, 85)

In [ ]:
# Plot the gauge locations colored by the maximum discharge (flooding severity)
plot_gauge_locations(ca_df, "Base Flow Index", "bfi_ave", "Base Flow Index (%)", 0, 100)

# Base Flow Index (BFI) is a ratio of base flow to total streamflow, expressed as a percentage and ranging 
# from 0 to 100. Base flow is the sustained, slowly varying component of streamflow, usually attributed to ground
# water discharge to a stream. 

Climate Attributes

In [ ]:
# Create table of average precipitation data
ppt_cols = [col for col in ca_df.columns if col.startswith("PPT") and col.endswith("_AVG")]
ppt_avg = ca_df[ppt_cols].mean()
years = [int(col[3:7]) for col in ppt_cols]

In [ ]:
# Calculate linear regression slope for precipitation
from scipy import stats

slope, intercept, r_value, p_value, std_err = stats.linregress(years, ppt_avg.values)
regression_line = [slope * y + intercept for y in years]

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(years, ppt_avg.values, label="Avg Precipitation")
ax.plot(years, regression_line, color="red", linestyle="--", label=f"Linear Fit (R²={r_value**2:.3f})")

equation = f"y = {slope:.4f}x + {intercept:.2f}"
ax.text(0.05, 0.95, equation, transform=ax.transAxes, verticalalignment="top")

ax.set_title("Average Annual Precipitation Across All Sites")
ax.set_xlabel("Year")
ax.set_ylabel("Average Precipitation (cm)")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Create table of average temperature data
tmp_cols = [col for col in ca_df.columns if col.startswith("TMP") and col.endswith("_AVG")]
tmp_avg = ca_df[tmp_cols].mean()
years = [int(col[3:7]) for col in tmp_cols]

In [ ]:
# Calculate linear regression slope for temperature

slope, intercept, r_value, p_value, std_err = stats.linregress(years, tmp_avg.values)
regression_line = [slope * y + intercept for y in years]

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(years, tmp_avg.values, label="Avg Temperature")
ax.plot(years, regression_line, color="red", linestyle="--", label=f"Linear Fit (R²={r_value**2:.3f})")

equation = f"y = {slope:.4f}x + {intercept:.2f}"
ax.text(0.05, 0.95, equation, transform=ax.transAxes, verticalalignment="top")

ax.set_title("Average Annual Temperature Across All Sites")
ax.set_xlabel("Year")
ax.set_ylabel("Average Temperature (°C)")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Create new Dataframe with average Precipitation by year for each HUC02 code.
ppt_cols = [col for col in ca_df.columns if col.startswith("PPT") and col.endswith("_AVG")]
ppt_by_huc = ca_df.groupby("HUC02")[ppt_cols].mean().reset_index()
ppt_by_huc.columns = ["HUC02"] + [col[3:7] for col in ppt_cols]

# print(ppt_by_huc.head())

In [ ]:


# Melt to long format: one row per (HUC02, year)
ppt_long = ppt_by_huc.melt(id_vars="HUC02", var_name="year", value_name="avg_ppt")
ppt_long["year"] = ppt_long["year"].astype(int)  

# Limit the sample to just these HUC Codes.
# ppt_long = ppt_long[ppt_long["HUC02"].isin(["10U", "10L"])]


fig, ax = plt.subplots(figsize=(12, 5))
for huc, group in ppt_long.groupby("HUC02"):
    group = group.sort_values("year")
    ax.plot(group["year"], group["avg_ppt"], label=huc)

ax.set_title("Average Annual Precipitation by HUC02 Region")
ax.set_xlabel("Year")
ax.set_ylabel("Average Precipitation (cm)")
ax.legend(title="HUC02", bbox_to_anchor=(1.01, 1), loc="upper left", borderaxespad=0)

plt.tight_layout()
plt.show()


In [ ]:
# Create new Dataframe with average temperature by year for each HUC02 code.
tmp_cols = [col for col in ca_df.columns if col.startswith("TMP") and col.endswith("_AVG")]
tmp_by_huc = ca_df.groupby("HUC02")[tmp_cols].mean().reset_index()
tmp_by_huc.columns = ["HUC02"] + [col[3:7] for col in tmp_cols]

# print(tmp_by_huc.head())

In [ ]:
# Melt to long format: one row per (HUC02, year)
tmp_long = tmp_by_huc.melt(id_vars="HUC02", var_name="year", value_name="avg_tmp")
tmp_long["year"] = tmp_long["year"].astype(int)  

# Limit the sample to just these HUC Codes.
# tmp_long = tmp_long[tmp_long["HUC02"].isin(["10U", "10L"])]


fig, ax = plt.subplots(figsize=(12, 5))
for huc, group in tmp_long.groupby("HUC02"):
    group = group.sort_values("year")
    ax.plot(group["year"], group["avg_tmp"], label=huc)

ax.set_title("Average Annual Temperature by HUC02 Region")
ax.set_xlabel("Year")
ax.set_ylabel("Average Temperature (°C)")
ax.legend(title="HUC02", bbox_to_anchor=(1.01, 1), loc="upper left", borderaxespad=0)
plt.tight_layout()
plt.show()

Soil Comparison

In [ ]:
soil_cols = ["clay_pct_avg", "silt_pct_avg", "sand_pct_avg"]
soil_by_huc = ca_df.groupby("HUC02")[soil_cols].mean()

fig, ax = plt.subplots(figsize=(10, 5))
soil_by_huc.plot(kind="bar", stacked=True, ax=ax, label=["Clay", "Silt", "Sand"])

ax.set_title("Soil Composition by HUC02 Region")
ax.set_xlabel("HUC02")
ax.set_ylabel("Percentage (%)")
ax.legend(["Clay", "Silt", "Sand"], title="Soil Type")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()
